In [6]:
## retrieval
## data ingestion

In [7]:
## data ingestion
## data loader, chunking, embeddings, store

In [8]:
import os
from dotenv import load_dotenv

In [9]:
load_dotenv()

True

In [10]:
from langchain_community.document_loaders import  TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# DATA_FILEPATH = os.path.join('data', 'hr_policy.txt')
DATA_FILEPATH = os.path.join("../../data" , "hr_policy.txt")

loader = TextLoader(DATA_FILEPATH, encoding='utf-8')
documents = loader.load()

len(documents)

1

In [11]:
documents[0].metadata

{'source': '../../data\\hr_policy.txt'}

In [12]:
print(f"Loaded file: {DATA_FILEPATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: ../../data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2598

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


In [13]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': '../../data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': '../../data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': '../../data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during

In [14]:
len(chunks)

9

In [15]:
## embeddings

In [16]:
from langchain_community.embeddings import JinaEmbeddings

embedding_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
embedding_model.model_name

'jina-embeddings-v2-base-en'

In [17]:
## storing
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding_model)


In [18]:
vector_store.index.ntotal ## total number of vectors stored

9

In [19]:
## retrieval

In [20]:
test_query = "How many sick leaves employees get"
top_matches = vector_store.similarity_search(test_query, k=3)
top_matches

[Document(id='a91db33f-3d48-4d82-8a34-b697a6d2506b', metadata={'source': '../../data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'),
 Document(id='679b0793-6fcc-4d12-8e31-fbb5abfd1bf5', metadata={'source': '../../data\\hr_policy.txt'}, page_content='7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.'),
 Document(id='6aee7182-3b33-4d3a-94cf-0f9b80e5f861', metadata={'source': '../../d

In [21]:
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

--- Match 3 ---
3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager a

In [22]:

retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(query:str) -> str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    
    retrieved_chunks = retriever.invoke(query)
    return "\n\n".join(chunk.page_content for chunk in retrieved_chunks)

In [23]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b")
llm.invoke("Hello").content

'Hello! 👋 How can I help you today?'

In [24]:
response = llm.invoke("tell me about code of conduct")
print(response.content)

## What is a Code of Conduct?

A **Code of Conduct** (CoC) is a formal set of rules, expectations, and guidelines that outline acceptable behavior within a community, organization, project, or event. It’s a living document that:

| Purpose | Why it matters |
|---------|----------------|
| **Create a safe, inclusive environment** | People feel respected and are more likely to contribute. |
| **Set clear expectations** | Reduces ambiguity about what’s acceptable. |
| **Provide a framework for conflict resolution** | Offers a consistent approach to handling incidents. |
| **Protect the community’s reputation** | Demonstrates professionalism and accountability. |
| **Encourage diversity & inclusion** | Signals that all voices are valued. |

---

## Typical Structure of a Code of Conduct

1. **Title & Scope**
   * Who it applies to (contributors, users, staff, event attendees, etc.).
2. **Purpose & Vision**
   * A short statement about the community’s values.
3. **Behavioral Expectations**


In [25]:
response.usage_metadata

{'input_tokens': 77,
 'output_tokens': 1454,
 'total_tokens': 1531,
 'output_token_details': {'reasoning': 117}}

In [26]:
response.response_metadata

{'token_usage': {'completion_tokens': 1454,
  'prompt_tokens': 77,
  'total_tokens': 1531,
  'completion_time': 1.715248421,
  'completion_tokens_details': {'reasoning_tokens': 117},
  'prompt_time': 0.003720224,
  'prompt_tokens_details': None,
  'queue_time': 0.046977306,
  'total_time': 1.718968645},
 'model_name': 'openai/gpt-oss-20b',
 'system_fingerprint': 'fp_e99e93f2ac',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}

In [27]:
## agents

In [28]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

In [29]:
system_prompt = """
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up facts before answering. 
    
    If the answer isn't in the search results, say you don't know "instead of guessing."
    
    always reply in 2 lines, if you are not clear ask a questions to clarift from user
    
    Dont entertain any other questions which are off topic and if you dont understand, tell that mail to acmecorp.com"""

In [30]:
hr_assitant = create_agent(
    model = llm,
    tools = [search_hr_policy],
    system_prompt=system_prompt
)

In [31]:
def ask_hr_assitant(query: str) -> str:
    print("user query: ",query)
    
    response = hr_assitant.invoke(
                    {
                        "messages": [
                            {
                                "role": "user",
                                "content": query
                            }
                        ]
                    }
                )

    answer = response["messages"][-1].content
    # print("response: ",answer)
    return answer


In [32]:
ask_hr_assitant("tell me about code of conduct")

user query:  tell me about code of conduct


'The code of conduct requires employees to maintain professionalism, respect, and prohibits harassment or discrimination. Violations may lead to disciplinary action, including termination, and all staff must complete annual training.'

In [33]:
ask_hr_assitant("What was my previous question")

user query:  What was my previous question


'Your previous question was “What was my previous question.”  \nLet me know if you need anything else.'

In [34]:
## add memory

In [35]:
system_prompt=""" 
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know 
    instead of guessing.
    
    Always reply in 2 lines, if you are not clear ask a questions to clarify from user.
    
    """

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

hr_assitant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt=system_prompt,
    checkpointer= InMemorySaver(),
)
config = {"configurable": {"thread_id": "user-session-10"}}

In [37]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


def ask_hr_assitant(query: str, config) -> str:
    print("user query: ",query)
    
    response = hr_assitant.invoke(
                    {
                        "messages": [
                            {
                                "role": "user",
                                "content": query
                            }
                        ]
                    }, config=config
                )

    answer = response["messages"][-1].content
    # print("response: ",answer)
    return answer


In [38]:
ask_hr_assitant("How to make a coffee", config)

user query:  How to make a coffee


'I’m sorry, I don’t know how to make coffee.  \nCould you clarify what you need—perhaps an office coffee policy or a recipe?'

In [39]:
ask_hr_assitant("What was my previous query", config)

user query:  What was my previous query


'Your previous query was: “How to make a coffee.”  \nLet me know if you’d like more details on that.'

In [40]:
# import sys

# print(sys.executable)
# !{sys.executable} -m ensurepip --upgrade
# !{sys.executable} -m pip install --upgrade pip
# !{sys.executable} -m pip install python-dotenv


# mkdir MyProject
# cd MyProject

# uv init

# uv venv

# uv add --dev ipykernel jupyter

# uv run python -m ipykernel install --user --name myproject --display-name "Python (MyProject)"

# uv add python-dotenv langchain langchain-community langchain-text-splitters

# uv sync

In [41]:
# Command                             Purpose

# uv venv                             Creates environment without pip
# uv venv --seed                      Creates environment with pip and other packaging tools
# uv add package                      Adds project dependency
# uv pip install package              Installs package using uv
# python -m pip install package       Installs using pip

In [42]:
# !uv add faiss-cpu